In [1]:
import os
import pandas as pd
import numpy as np

os.chdir(r"D:\Projects\berlin-electricity-temperature")

# Load the raw temperature file we downloaded earlier
temp_raw = pd.read_csv("data/raw/dwd_temperature/berlin_tempelhof_raw.csv")

print("Shape:", temp_raw.shape)
print("Columns:", list(temp_raw.columns))
print(temp_raw.head(3))

Shape: (570247, 6)
Columns: ['STATIONS_ID', 'MESS_DATUM', 'QN_9', 'TT_TU', 'RF_TU', 'eor']
   STATIONS_ID  MESS_DATUM  QN_9  TT_TU  RF_TU  eor
0          433  1951010101     5  -10.7   88.0  eor
1          433  1951010102     5  -11.0   87.0  eor
2          433  1951010103     5  -11.2   87.0  eor


In [2]:
# Keep only temperature and humidity columns
temp = temp_raw[['MESS_DATUM', 'TT_TU', 'RF_TU']].copy()

# Rename to simple names
temp.columns = ['datetime', 'temperature_c', 'humidity_pct']

print("Shape:", temp.shape)
print(temp.head(3))

Shape: (570247, 3)
     datetime  temperature_c  humidity_pct
0  1951010101          -10.7          88.0
1  1951010102          -11.0          87.0
2  1951010103          -11.2          87.0


In [5]:
# DWD uses -999 to represent missing temperature readings
# We replace -999 with NaN (pandas standard for missing)

temp['temperature_c'] = temp['temperature_c'].replace(-999, np.nan)
temp['humidity_pct'] = temp['humidity_pct'].replace(-999, np.nan)

# Check how many missing values we have
missing_temp = temp['temperature_c'].isna().sum()
missing_hum  = temp['humidity_pct'].isna().sum()
total        = len(temp)

print(f"Missing temperature : {missing_temp} ({missing_temp/total*100:.2f}%)")
print(f"Missing humidity    : {missing_hum} ({missing_hum/total*100:.2f}%)")

Missing temperature : 342 (0.06%)
Missing humidity    : 691 (0.12%)


In [7]:
import os
import pandas as pd
import numpy as np

os.chdir(r"D:\Projects\berlin-electricity-temperature")

# Load raw file
temp_raw = pd.read_csv("data/raw/dwd_temperature/berlin_tempelhof_raw.csv")

# Keep only what we need
temp = temp_raw[['MESS_DATUM', 'TT_TU', 'RF_TU']].copy()
temp.columns = ['datetime', 'temperature_c', 'humidity_pct']

# Fix date format
temp['datetime'] = pd.to_datetime(temp['datetime'], format='%Y%m%d%H')

# Replace DWD missing value code with NaN
temp['temperature_c'] = temp['temperature_c'].replace(-999, np.nan)
temp['humidity_pct']  = temp['humidity_pct'].replace(-999, np.nan)

# Fill small gaps using interpolation
temp['temperature_c'] = temp['temperature_c'].interpolate(method='linear', limit=3)
temp['humidity_pct']  = temp['humidity_pct'].interpolate(method='linear', limit=3)

# Filter to study period
temp = temp[temp['datetime'] >= '2015-01-01'].reset_index(drop=True)

# Final check
print("Shape:", temp.shape)
print("First date:", temp['datetime'].iloc[0])
print("Last date:", temp['datetime'].iloc[-1])
print("Missing temperature:", temp['temperature_c'].isna().sum())
print("Missing humidity:", temp['humidity_pct'].isna().sum())
print(temp.head(3))

Shape: (97732, 3)
First date: 2015-01-01 00:00:00
Last date: 2026-02-26 23:00:00
Missing temperature: 301
Missing humidity: 541
             datetime  temperature_c  humidity_pct
0 2015-01-01 00:00:00            5.2          95.0
1 2015-01-01 01:00:00            5.1          95.0
2 2015-01-01 02:00:00            4.8          94.0


In [10]:
# Load the saved energy data
energy = pd.read_csv("data/processed/energy_clean.csv")
energy['datetime'] = pd.to_datetime(energy['datetime'])

print("Energy loaded:", energy.shape)
print(energy.head(2))

Energy loaded: (97789, 6)
             datetime demand_mwh wind_offshore_mwh wind_onshore_mwh solar_mwh  \
0 2015-01-01 00:00:00  44,600.25            516.50         8,128.00      0.00   
1 2015-01-01 01:00:00  43,454.75            516.25         8,297.50      0.00   

    gas_mwh  
0  1,226.25  
1    870.75  


In [11]:
# Now merge temperature with energy
master = pd.merge(energy, temp, on='datetime', how='inner')

print("Shape:", master.shape)
print("Columns:", list(master.columns))
print(master.head(3))

Shape: (97721, 8)
Columns: ['datetime', 'demand_mwh', 'wind_offshore_mwh', 'wind_onshore_mwh', 'solar_mwh', 'gas_mwh', 'temperature_c', 'humidity_pct']
             datetime demand_mwh wind_offshore_mwh wind_onshore_mwh solar_mwh  \
0 2015-01-01 00:00:00  44,600.25            516.50         8,128.00      0.00   
1 2015-01-01 01:00:00  43,454.75            516.25         8,297.50      0.00   
2 2015-01-01 02:00:00  41,963.25            514.00         8,540.00      0.00   

    gas_mwh  temperature_c  humidity_pct  
0  1,226.25            5.2          95.0  
1    870.75            5.1          95.0  
2    809.50            4.8          94.0  
